# 05 — Exportar el modelo a ONNX y calibrar el umbral (#38)

**Qué hace:** convierte el modelo del #36 a **ONNX int8**, el formato que usará el backend sin torch, y elige el **umbral de fraud** con el conjunto de validación.

**Qué necesitas:**
1. El zip del modelo del #36 (`avizor-3c.zip`). Si lo guardaste en tu Google Drive (Mi unidad), el notebook lo encuentra solo. Si no, te lo pide.
2. Los mismos dos CSV del entrenamiento: `Spaphish dataset - DiB.csv` y `spamspa_review.csv`.

**Cómo usarlo:** **Entorno de ejecución → Ejecutar todo**. No necesita GPU. Al terminar se descarga `avizor-3c-onnx.zip`.

Los datos subidos se borran al cerrar la sesión de Colab. No se suben a GitHub.

**1. Descargamos el código de la rama del #38 e instalamos lo que falta.**

In [ ]:
BRANCH = "feat/146-retrain-synthetic"

!git clone -q -b {BRANCH} https://github.com/MariaIsaDurango/Avizor.git
%cd Avizor/ml
!pip install -q "transformers>=4.46" onnx onnxscript onnxruntime

**2. Descargamos SpamSpa de HuggingFace** (el mismo `train.csv` del entrenamiento).

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

RAW = Path("data/raw")
(RAW / "spamspa").mkdir(parents=True, exist_ok=True)
(RAW / "spaphish").mkdir(parents=True, exist_ok=True)

spamspa_path = hf_hub_download(
    repo_id="softecapps/spam_ham_spanish",
    filename="train.csv",
    repo_type="dataset",
    local_dir=RAW / "spamspa",
)
print("SpamSpa:", spamspa_path)

**3. Traemos el modelo del #36.** Primero lo busca en tu Drive; si no está, pulsa *Elegir archivos* y sube `avizor-3c.zip`.

In [ ]:
import json
import zipfile
from google.colab import drive, files


def model_version(path):
    with zipfile.ZipFile(path) as archive:
        return json.loads(archive.read("metrics.json"))["model_version"]


# ES: Solo vale el modelo BETO del #36 (los zips "roberta" son de pruebas viejas).
drive.mount("/content/drive")
found = [
    p for p in sorted(Path("/content/drive/MyDrive").glob("avizor*3c*.zip"))
    if "onnx" not in p.name and model_version(p).startswith("beto-3c")
]
if found:
    model_zip = found[-1]
    print("Modelo encontrado en Drive:", model_zip)
else:
    uploaded = files.upload()
    model_zip = Path("/content") / next(iter(uploaded))
    model_zip.write_bytes(uploaded[model_zip.name])

print("Versión:", model_version(model_zip))
assert model_version(model_zip).startswith("beto-3c"), "Este zip no es el modelo BETO del #36"
MODEL = Path("models/avizor-3c")
zipfile.ZipFile(model_zip).extractall(MODEL)

**4. Subimos los dos CSV locales:**
- `Spaphish dataset - DiB.csv`
- `spamspa_review.csv` (la misma revisión usada para entrenar)

In [ ]:
uploaded = files.upload()
for name, content in uploaded.items():
    target = RAW / ("spaphish" if "paphish" in name.lower() else "spamspa") / name
    target.write_bytes(content)
    print("Guardado:", target)

SPAPHISH = next((RAW / "spaphish").glob("*.csv"))
REVIEW = next((RAW / "spamspa").glob("*review*.csv"))
print(SPAPHISH, REVIEW, sep="\n")

**5. Exportamos, cuantizamos y calibramos.** Tarda unos 5 minutos. Si los CSV no son los del entrenamiento, se detiene con un aviso.

In [ ]:
!python -m src.classifier.export \
    --model-dir models/avizor-3c/model \
    --spaphish "{SPAPHISH}" \
    --spamspa "{spamspa_path}" \
    --review "{REVIEW}" \
    --synthetic data/raw/smishing_groq.csv \
    --synthetic data/raw/legit_links_groq.csv \
    --out models/avizor-3c-onnx

**6. Resultados.** Compara el test sin umbral (como en el #36) y con el umbral calibrado.

In [ ]:
import json

report = json.loads(Path("models/avizor-3c-onnx/metrics_onnx.json").read_text())
print("Versión:", report["model_version"])
print("Umbral de fraud:", report["fraud_threshold"])
print("Tamaño (MB):", report["size_mb"])
print("Latencia CPU (ms):", {k: round(v) for k, v in report["cpu_latency_ms_median"].items()})
print("Coincidencia int8 vs fp32:", round(report["int8_vs_fp32_agreement"], 3))
for name in ("test_argmax_int8", "test_threshold_int8"):
    print(f"\n{name}:", report[name]["targets"])
    for source, values in report[name]["by_source"].items():
        keys = ("fraud_precision", "fraud_recall", "suspicious_recall", "accuracy")
        print(f"  {source}:", {k: round(values[k], 3) for k in keys})

**7. Descargamos el paquete para el backend** y guardamos una copia en Drive.

In [ ]:
!cd models/avizor-3c-onnx && zip -qr /content/avizor-3c-onnx.zip .
!cp /content/avizor-3c-onnx.zip /content/drive/MyDrive/
files.download("/content/avizor-3c-onnx.zip")